In [1]:
import pandas as pd
import os
import kagglehub
from dotenv import load_dotenv
import numpy as np

c:\Users\kimmi\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Load .env file for Kaggle Credentials 
load_dotenv()
username = os.getenv("KAGGLE_USERNAME")
key = os.getenv("KAGGLE_KEY")

os.environ["KAGGLE_USERNAME"] = username
os.environ["KAGGLE_KEY"] = key

In [3]:
#Locate path to the Kaggle dataset
path = kagglehub.dataset_download(
    "aiaiaidavid/the-big-dataset-of-ultra-marathon-running"
)

In [4]:
df = pd.read_csv(f"{path}/TWO_CENTURIES_OF_UM_RACES.csv", encoding="latin1")
df = df.drop(columns=["Event dates", "Event number of finishers", "Athlete club", "Athlete average speed"])

C:\Users\kimmi\AppData\Local\Temp\ipykernel_56664\1301004287.py:1: DtypeWarning: Columns (11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f"{path}/TWO_CENTURIES_OF_UM_RACES.csv", encoding="latin1")


In [5]:
#1) Extract data since 2000s (TRY DIFFERENT THRESHOLDS)
df = df[df['Year of event'] >= 2000]

In [6]:
#2) Extract athletes that have data on a large portion of their career
threshold_of_races = 15
valid_athletes = df['Athlete ID'].value_counts()
valid_athletes = valid_athletes[valid_athletes > threshold_of_races].index
experienced_df = df[df['Athlete ID'].isin(valid_athletes)]
experienced_df['Athlete ID'].nunique()

81603

In [7]:
#Feature of race_count --> Number of times a unique ID appears
experienced_df = df[df['Athlete ID'].isin(valid_athletes)].copy()
experienced_df["race_count"] = experienced_df.groupby("Athlete ID")["Athlete ID"].transform("count")

In [8]:
#Count the number of entries for each race
experienced_df['Event name'].value_counts()

Event name
Two Oceans Marathon (RSA)                                    66695
Comrades Marathon - Down Run (RSA)                           49422
Comrades Marathon - Up Run (RSA)                             45381
Two Oceans Marathon - 50km Split (RSA)                       38182
Ultra Trail Tour du Mont Blanc (UTMB) (FRA)                  15324
                                                             ...  
Universiti Malaya 6 hours Ultra Marathon (MAS)                   1
Ranscombe Autumn Challenge 8 hours run - Sunday Run (GBR)        1
Buckley's Chance 50K Trail Run (AUS)                             1
Javadhu Hills Ultra 50 Km (IND)                                  1
100 Miles Thailand Back2Back (THA)                               1
Name: count, Length: 23638, dtype: int64

In [9]:
df_two_oceans = experienced_df[experienced_df['Event name'] == 'Two Oceans Marathon (RSA)'].copy()
df_two_oceans.dropna()
print(f"Total athletes: {df_two_oceans["Athlete ID"].nunique()}")
print(f"Total entries: {len(df_two_oceans["Athlete ID"])}")

Total athletes: 10447
Total entries: 66695


In [10]:
#Feature two_oceans_count --> Number of times they have ran this event
df_two_oceans["two_oceans_count"] = df_two_oceans.groupby("Athlete ID")["Athlete ID"].transform("count")
df_two_oceans["two_oceans_count"]

171456      5
171458      9
171462      9
171463      9
171465      9
           ..
6360620     6
6360630    16
6360638    13
6360639    14
6360648     7
Name: two_oceans_count, Length: 66695, dtype: int64

In [11]:
#Calculate the athletes average speed in km/hr --> Used to Extract top threshold of fastest runners
#Since format H:MM:SS h harder to compare, numerically

df_two_oceans["distance_km"] = df_two_oceans["Event distance/length"].str.extract(r"(\d+\.?\d*)").astype(float)
df_two_oceans["distance_km"]
time_parts = df_two_oceans["Athlete performance"].str.replace(" h", "").str.split(":", expand=True)

df_two_oceans["hours"] = (
    time_parts[0].astype(float) +
    time_parts[1].astype(float) / 60 +
    time_parts[2].astype(float) / 3600
)

df_two_oceans["calculated_speed"] = df_two_oceans["distance_km"] / df_two_oceans["hours"]
df_two_oceans["calculated_speed"].head()

171456    17.571690
171458    17.486339
171462    17.241084
171463    17.058724
171465    16.959704
Name: calculated_speed, dtype: float64

In [12]:
#Filter the top 5% of the fastest athletes
# adding feature for 'elite' runners, top 15% threshold can be adjusted
# df_two_oceans['Elite'] = df_two_oceans['Calculated speed'] >= df_two_oceans['Calculated speed'].quantile(0.95)  # per race or per year
threshold = df_two_oceans["calculated_speed"].quantile(0.90)

df_top10 = df_two_oceans[df_two_oceans["calculated_speed"] >= threshold].copy()

df_top10["finish_time_mins"] = df_top10["hours"] * 60

print(f"Total Entries: {len(df_two_oceans)}")
print(f"Top 10% Entries: {len(df_top10)}\n")
print(df_top10["calculated_speed"].describe())


Total Entries: 66695
Top 10% Entries: 6670

count    6670.000000
mean       13.724108
std         1.175866
min        12.330275
25%        12.754045
50%        13.439104
75%        14.328104
max        18.096948
Name: calculated_speed, dtype: float64


In [13]:
# runners are overwhelmingly from South Africa for this race
# potential domestic/foreign feature??
df_two_oceans["Athlete country"].value_counts()

Athlete country
RSA    63623
GBR      751
GER      444
ZIM      265
NAM      261
LES      174
USA      132
SWZ      101
AUS       95
CAN       86
NED       84
FRA       63
IRL       54
SUI       51
BOT       49
POR       43
MAW       40
BEL       36
ITA       34
SWE       32
AUT       28
RUS       26
CZE       22
ZAM       21
BRA       14
NZL       14
IND       14
POL       13
BLR       12
UGA       11
MOZ        8
KEN        8
COD        8
NOR        8
ISR        7
HUN        7
JPN        6
CMR        6
TPE        5
DEN        5
MAS        4
BEN        4
CHI        3
ANG        3
UAE        3
LUX        3
ARG        2
HKG        2
IVB        1
FIN        1
XXX        1
SVK        1
PHI        1
LAT        1
SGP        1
UKR        1
URU        1
ROU        1
Name: count, dtype: int64

In [ ]:
# 1. Athlete Age
df_top10["athlete_age"] = df_top10["Year of event"] - df_top10["Athlete year of birth"]

# 2. Age squared - captures non-linear age effect (performance peaks then declines)
df_top10["athlete_age_squared"] = df_top10["athlete_age"] ** 2

# 3. Gender - binary encode
df_top10["gender_male"] = (df_top10["Athlete gender"] == "M").astype(int)

# 4. Domestic vs International
df_top10["is_domestic"] = (df_top10["Athlete country"] == "RSA").astype(int)

# 5. Age from peak (marathon peak age ~28-32, using 30 as midpoint)
df_top10["age_from_peak"] = abs(df_top10["athlete_age"] - 30)

# 6. Experience - log transform to diminish returns of extra races
# log(1 + x) ensures athletes with 0 previous races don't cause log(0)
df_top10["experience_log"] = np.log1p(df_top10["race_count"] * df_top10["two_oceans_count"])

# 7. Age x Gender interaction
df_top10["age_gender"] = df_top10["athlete_age"] * df_top10["gender_male"]

# 8. Age x log(experience) interaction
df_top10["age_x_experience"] = df_top10["athlete_age"] * df_top10["experience_log"]

In [ ]:
features = [
    "athlete_age",
    "athlete_age_squared",
    "gender_male",
    "is_domestic",
    "age_from_peak",
    "experience_log",
    "age_gender",
    "age_x_experience",
]

target = "finish_time_mins"

# 1. Correlation with TARGET
target_corr = df_top10[features + ["finish_time_mins"]].corr()["finish_time_mins"].sort_values()
print(target_corr)

# 2. Feature-to-feature correlation — multicollinearity check  
feature_corr = df_top10[features].corr()
print(feature_corr)

experience_log         -0.228702
age_x_experience       -0.147336
performance_std        -0.115227
gender_male            -0.060162
improvement_trend       0.058210
age_gender              0.105106
is_domestic             0.169988
athlete_age             0.239082
athlete_age_squared     0.239524
age_from_peak           0.241211
performance_vs_field    0.997541
finish_time_mins        1.000000
Name: finish_time_mins, dtype: float64
                      athlete_age  athlete_age_squared  gender_male  \
athlete_age              1.000000             0.992451     0.014901   
athlete_age_squared      0.992451             1.000000     0.018624   
gender_male              0.014901             0.018624     1.000000   
is_domestic             -0.030983            -0.031233     0.117051   
age_from_peak            0.952140             0.969988     0.020700   
experience_log           0.112329             0.110689     0.041520   
age_gender               0.647648             0.646471     0.761179 

In [16]:
import numpy as np
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

In [17]:
model_df = df_top10[features + [target, "Year of event"]].dropna()

In [18]:
#TIME BASED MODEL SPLIT
train_df = model_df[model_df["Year of event"] <= 2015]
val_df   = model_df[(model_df["Year of event"] >= 2016) & (model_df["Year of event"] <= 2017)]
test_df  = model_df[model_df["Year of event"] >= 2018]

In [19]:
X_train, y_train = train_df[features], train_df[target]
X_val,   y_val   = val_df[features],   val_df[target]
X_test,  y_test  = test_df[features],  test_df[target]

In [20]:
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

Train: 3021 | Val: 0 | Test: 0


In [21]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit + transform
X_val_s   = scaler.transform(X_val)         # transform only
X_test_s  = scaler.transform(X_test)        # transform only

ValueError: Found array with 0 sample(s) (shape=(0, 11)) while a minimum of 1 is required by StandardScaler.

In [ ]:
def evaluate(name, y_true, y_pred):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2   = r2_score(y_true, y_pred)
    print(f"\n{'─'*40}")
    print(f"  {name}")
    print(f"  MAE:  {mae:.2f} mins")
    print(f"  RMSE: {rmse:.2f} mins")
    print(f"  R²:   {r2:.4f}")
    return {"model": name, "MAE": mae, "RMSE": rmse, "R2": r2}

results = []

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 1 — LINEAR REGRESSION (Ridge for stability)
# ══════════════════════════════════════════════════════════════════════════════
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_s, y_train)

# Tune alpha on validation set
best_alpha, best_val_mae = 1.0, float("inf")
for alpha in [0.01, 0.1, 1, 10, 100]:
    m = Ridge(alpha=alpha).fit(X_train_s, y_train)
    mae = mean_absolute_error(y_val, m.predict(X_val_s))
    if mae < best_val_mae:
        best_val_mae, best_alpha = mae, alpha

ridge_final = Ridge(alpha=best_alpha).fit(X_train_s, y_train)
results.append(evaluate("Ridge Regression", y_test, ridge_final.predict(X_test_s)))

# Coefficients — interpretability
coef_df = pd.DataFrame({"feature": features, "coef": ridge_final.coef_})
print(coef_df.sort_values("coef", key=abs, ascending=False))


────────────────────────────────────────
  Ridge Regression
  MAE:  9.68 mins
  RMSE: 12.35 mins
  R²:   0.5714
                 feature      coef
9          rolling_avg_3  6.331679
8       prev_finish_time  4.865437
11  personal_best_so_far  3.899847
0            athlete_age  1.279225
10       field_avg_speed -1.190845
7       age_x_experience  1.182108
3            is_domestic  0.955911
2            gender_male -0.559053
5         experience_log  0.526514
6             age_gender  0.445890
1    athlete_age_squared  0.281575
4          age_from_peak  0.221685


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 2 — REGRESSION TREE
# ══════════════════════════════════════════════════════════════════════════════
# Tune max_depth via validation MAE
best_depth, best_val_mae = 3, float("inf")
for depth in range(2, 12):
    m = DecisionTreeRegressor(max_depth=depth, random_state=42).fit(X_train, y_train)
    mae = mean_absolute_error(y_val, m.predict(X_val))
    if mae < best_val_mae:
        best_val_mae, best_depth = mae, depth

tree_final = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
tree_final.fit(X_train, y_train)   # Trees don't need scaling
results.append(evaluate(f"Regression Tree (depth={best_depth})", y_test, tree_final.predict(X_test)))

# Feature importances
imp_df = pd.DataFrame({"feature": features, "importance": tree_final.feature_importances_})
print(imp_df.sort_values("importance", ascending=False))


────────────────────────────────────────
  Regression Tree (depth=5)
  MAE:  9.70 mins
  RMSE: 12.47 mins
  R²:   0.5633
                 feature  importance
9          rolling_avg_3    0.927894
8       prev_finish_time    0.036457
4          age_from_peak    0.007461
7       age_x_experience    0.007058
6             age_gender    0.006846
1    athlete_age_squared    0.003892
0            athlete_age    0.003886
5         experience_log    0.003029
11  personal_best_so_far    0.002183
10       field_avg_speed    0.001294
3            is_domestic    0.000000
2            gender_male    0.000000


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 3 — GRADIENT BOOSTING (sklearn GBM)
# ══════════════════════════════════════════════════════════════════════════════
# Tune n_estimators & learning_rate on validation set
best_params, best_val_mae = {}, float("inf")
for n_est in [100, 200, 300]:
    for lr in [0.01, 0.05, 0.1]:
        m = GradientBoostingRegressor(
            n_estimators=n_est, learning_rate=lr,
            max_depth=4, subsample=0.8, random_state=42
        ).fit(X_train, y_train)
        mae = mean_absolute_error(y_val, m.predict(X_val))
        if mae < best_val_mae:
            best_val_mae, best_params = mae, {"n_estimators": n_est, "learning_rate": lr}

gbm_final = GradientBoostingRegressor(**best_params, max_depth=4, subsample=0.8, random_state=42)
gbm_final.fit(X_train, y_train)
results.append(evaluate(f"Gradient Boosting {best_params}", y_test, gbm_final.predict(X_test)))


────────────────────────────────────────
  Gradient Boosting {'n_estimators': 100, 'learning_rate': 0.1}
  MAE:  9.36 mins
  RMSE: 12.36 mins
  R²:   0.5707
